# K2 annual harmonics: build all VRTs first, then write all derived monolith TIFFs

This notebook follows the same core workflow used in the earlier harmonic-sensitivity notebook:

- discover/download GCS shards;
- QA source shards with `rasterio`;
- build a plain spatial VRT with `gdalbuildvrt`;
- use the VRT as the read source;
- write the final derived GeoTIFF window-by-window with `rasterio`.

The workflow is deliberately split into two phases.

## Phase 1 — all years

For every year 2016–2025:

1. inventory the correct GCS prefix;
2. download the 9 GeoTIFF shards to `C:\NCA_DATA\S2_Harmonics\YYYY`;
3. verify source band count / CRS / resolution / NoData / band names;
4. build `YYYY_K2.vrt`;
5. verify that the VRT opens correctly.

**No large derived TIFF is written until all yearly VRTs are complete.**

## Phase 2 — all years

For every completed VRT:

1. copy intercept (`constant_*`) bands;
2. convert each K2 sine/cosine pair to:
   - amplitude = `sqrt(sin² + cos²)`
   - phase = `atan2(sin, cos)`
3. copy `nobs`;
4. copy the per-variable RMSE bands;
5. write one 91-band float32 monolith:
   - `YYYY_K2_InterceptAmpPhase_nobs_RMSE.tif`
6. QA the final band map;
7. copy the completed year folder to:
   - `A:\NCA_DATA\S2_Harmonics\YYYY`

The source shards and VRT remain in the year folder so each annual product is self-contained and reproducible.


In [6]:
from pathlib import Path
import subprocess
import shutil
import os

import numpy as np
import pandas as pd
import rasterio

# -------------------------------------------------------------------
# LOCAL / CLOUD CONFIGURATION
# -------------------------------------------------------------------

WORK_ROOT = Path(r"C:\NCA_DATA\S2_Harmonics")
ARCHIVE_ROOT = Path(r"A:\NCA_DATA\S2_Harmonics")

GCLOUD_CMD = Path(
    r"C:\Users\scottfordham\AppData\Local\Google\Cloud SDK"
    r"\google-cloud-sdk\bin\gcloud.cmd"
)

assert GCLOUD_CMD.exists(), f"gcloud not found: {GCLOUD_CMD}"

GCS_BUCKET = "bop-nca-data-space-s2-c1-staging"

YEARS = list(range(2016, 2026))

K = 2
NODATA = -32768.0
EXPECTED_SOURCE_BANDS = 91
EXPECTED_OUTPUT_BANDS = 91

VARIABLES = [
    "B2", "B3", "B4", "B5", "B6",
    "B7", "B8", "B8A", "B11", "B12",
    "NDVI", "NDMI", "NDRE", "BSI", "NBR2",
]

def gcs_prefix_for_year(year):
    if year in (2016, 2017):
        return f"harmonics_k2/annual_2016_2017/{year}"
    if 2018 <= year <= 2025:
        return f"harmonics_k2/annual_2018_2025/{year}"
    raise ValueError(year)

def export_prefix_for_year(year):
    return f"S2SR_Harmonics_{year}_K2"

WORK_ROOT.mkdir(parents=True, exist_ok=True)
ARCHIVE_ROOT.mkdir(parents=True, exist_ok=True)

print("Working root:", WORK_ROOT)
print("Archive root:", ARCHIVE_ROOT)
print("NoData:", NODATA)

for year in YEARS:
    print(
        year,
        "->",
        f"gs://{GCS_BUCKET}/{gcs_prefix_for_year(year)}/"
    )


Working root: C:\NCA_DATA\S2_Harmonics
Archive root: A:\NCA_DATA\S2_Harmonics
NoData: -32768.0
2016 -> gs://bop-nca-data-space-s2-c1-staging/harmonics_k2/annual_2016_2017/2016/
2017 -> gs://bop-nca-data-space-s2-c1-staging/harmonics_k2/annual_2016_2017/2017/
2018 -> gs://bop-nca-data-space-s2-c1-staging/harmonics_k2/annual_2018_2025/2018/
2019 -> gs://bop-nca-data-space-s2-c1-staging/harmonics_k2/annual_2018_2025/2019/
2020 -> gs://bop-nca-data-space-s2-c1-staging/harmonics_k2/annual_2018_2025/2020/
2021 -> gs://bop-nca-data-space-s2-c1-staging/harmonics_k2/annual_2018_2025/2021/
2022 -> gs://bop-nca-data-space-s2-c1-staging/harmonics_k2/annual_2018_2025/2022/
2023 -> gs://bop-nca-data-space-s2-c1-staging/harmonics_k2/annual_2018_2025/2023/
2024 -> gs://bop-nca-data-space-s2-c1-staging/harmonics_k2/annual_2018_2025/2024/
2025 -> gs://bop-nca-data-space-s2-c1-staging/harmonics_k2/annual_2018_2025/2025/


# Phase 1 — inventory, download, QA, and build **all VRTs**

This uses the same simple VRT construction pattern as the earlier K3 notebook:

```python
gdalbuildvrt -overwrite output.vrt shard1.tif shard2.tif ...
```

We do not rewrite band descriptions into the VRT. The authoritative band schema is read from a source shard and retained separately for the derived TIFF write.


In [7]:
# =====================================================================
# INVENTORY ALL YEARLY GCS PREFIXES
# =====================================================================

inventory_rows = []

for year in YEARS:

    gcs_root = (
        f"gs://{GCS_BUCKET}/"
        f"{gcs_prefix_for_year(year).rstrip('/')}/"
    )

    print(f"\nInventorying {year}:")
    print(gcs_root)

    cmd = [
        str(GCLOUD_CMD),
        "storage",
        "ls",
        "--recursive",
        gcs_root,
    ]

    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        check=False,
    )

    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f"{year}: gcloud storage ls failed")

    objects = [
        line.strip()
        for line in result.stdout.splitlines()
        if line.strip().startswith("gs://")
    ]

    export_prefix = export_prefix_for_year(year)

    matches = [
        uri
        for uri in objects
        if uri.rsplit("/", 1)[-1].startswith(export_prefix)
        and uri.lower().endswith(".tif")
    ]

    print("  matching TIFF shards:", len(matches))

    for uri in matches:
        inventory_rows.append({
            "year": year,
            "gcs_uri": uri,
            "filename": uri.rsplit("/", 1)[-1],
            "source_prefix": gcs_prefix_for_year(year),
        })

inventory = pd.DataFrame(inventory_rows)

display(
    inventory.groupby("year")
    .agg(
        shards=("filename", "size"),
        source_prefix=("source_prefix", "first"),
    )
    .reindex(YEARS)
)

if inventory.empty:
    raise RuntimeError("No K2 shards were found.")

missing_years = [
    year for year in YEARS
    if year not in set(inventory["year"])
]

if missing_years:
    raise RuntimeError(f"Missing years in inventory: {missing_years}")

bad_counts = (
    inventory.groupby("year")
    .size()
    .loc[lambda x: x != 9]
)

if len(bad_counts):
    raise RuntimeError(
        "Expected 9 shards per year. Unexpected counts:\n"
        + bad_counts.to_string()
    )

print("\nInventory QA passed: 9 shards for every year.")



Inventorying 2016:
gs://bop-nca-data-space-s2-c1-staging/harmonics_k2/annual_2016_2017/2016/
  matching TIFF shards: 9

Inventorying 2017:
gs://bop-nca-data-space-s2-c1-staging/harmonics_k2/annual_2016_2017/2017/
  matching TIFF shards: 9

Inventorying 2018:
gs://bop-nca-data-space-s2-c1-staging/harmonics_k2/annual_2018_2025/2018/
  matching TIFF shards: 9

Inventorying 2019:
gs://bop-nca-data-space-s2-c1-staging/harmonics_k2/annual_2018_2025/2019/
  matching TIFF shards: 9

Inventorying 2020:
gs://bop-nca-data-space-s2-c1-staging/harmonics_k2/annual_2018_2025/2020/
  matching TIFF shards: 9

Inventorying 2021:
gs://bop-nca-data-space-s2-c1-staging/harmonics_k2/annual_2018_2025/2021/
  matching TIFF shards: 9

Inventorying 2022:
gs://bop-nca-data-space-s2-c1-staging/harmonics_k2/annual_2018_2025/2022/
  matching TIFF shards: 9

Inventorying 2023:
gs://bop-nca-data-space-s2-c1-staging/harmonics_k2/annual_2018_2025/2023/
  matching TIFF shards: 9

Inventorying 2024:
gs://bop-nca-data-sp

,shards,source_prefix
year,,
2016,9,harmonics_k2/annual_2016_2017/2016
2017,9,harmonics_k2/annual_2016_2017/2017
2018,9,harmonics_k2/annual_2018_2025/2018
2019,9,harmonics_k2/annual_2018_2025/2019
2020,9,harmonics_k2/annual_2018_2025/2020
2021,9,harmonics_k2/annual_2018_2025/2021
2022,9,harmonics_k2/annual_2018_2025/2022
2023,9,harmonics_k2/annual_2018_2025/2023
2024,9,harmonics_k2/annual_2018_2025/2024



Inventory QA passed: 9 shards for every year.


In [8]:
# =====================================================================
# DOWNLOAD ALL SHARDS TO C:\NCA_DATA\S2_Harmonics\YYYY
# =====================================================================

local_shards_by_year = {}

for year in YEARS:

    year_dir = WORK_ROOT / str(year)
    year_dir.mkdir(parents=True, exist_ok=True)

    year_inventory = inventory.loc[
        inventory["year"].eq(year)
    ].copy()

    print(f"\n{year}: downloading/checking {len(year_inventory)} shards")

    for uri in year_inventory["gcs_uri"]:

        filename = uri.rsplit("/", 1)[-1]
        destination = year_dir / filename

        if destination.exists():
            print("Already present:", filename)
            continue

        print("Downloading:", filename)

        cmd = [
            str(GCLOUD_CMD),
            "storage",
            "cp",
            uri,
            str(destination),
        ]

        subprocess.run(
            cmd,
            check=True,
        )

    export_prefix = export_prefix_for_year(year)

    local_shards = sorted(
        year_dir.glob(f"{export_prefix}*.tif")
    )

    if len(local_shards) != len(year_inventory):
        raise RuntimeError(
            f"{year}: cloud/local shard mismatch: "
            f"{len(year_inventory)} cloud vs {len(local_shards)} local"
        )

    local_shards_by_year[year] = local_shards

print("\nAll yearly shard downloads are complete.")



2016: downloading/checking 9 shards
Already present: S2SR_Harmonics_2016_K20000000000-0000000000.tif
Already present: S2SR_Harmonics_2016_K20000000000-0000003584.tif
Already present: S2SR_Harmonics_2016_K20000000000-0000007168.tif
Already present: S2SR_Harmonics_2016_K20000003584-0000000000.tif
Already present: S2SR_Harmonics_2016_K20000003584-0000003584.tif
Already present: S2SR_Harmonics_2016_K20000003584-0000007168.tif
Already present: S2SR_Harmonics_2016_K20000007168-0000000000.tif
Already present: S2SR_Harmonics_2016_K20000007168-0000003584.tif
Already present: S2SR_Harmonics_2016_K20000007168-0000007168.tif

2017: downloading/checking 9 shards
Downloading: S2SR_Harmonics_2017_K20000000000-0000000000.tif
Downloading: S2SR_Harmonics_2017_K20000000000-0000003584.tif
Downloading: S2SR_Harmonics_2017_K20000000000-0000007168.tif
Downloading: S2SR_Harmonics_2017_K20000003584-0000000000.tif
Downloading: S2SR_Harmonics_2017_K20000003584-0000003584.tif
Downloading: S2SR_Harmonics_2017_K20

In [10]:
# Inspect the complete band list for one 2016 shard

year = 2016
shard = local_shards_by_year[year][0]

print("Shard:")
print(shard)

with rasterio.open(shard) as src:
    print("\nBand count:", src.count)
    print("CRS:", src.crs)
    print("NoData:", src.nodata)
    print("\nBand descriptions:\n")

    for i, desc in enumerate(src.descriptions, start=1):
        print(f"{i:3d}: {desc}")

Shard:
C:\NCA_DATA\S2_Harmonics\2016\S2SR_Harmonics_2016_K20000000000-0000000000.tif

Band count: 91
CRS: PROJCS["NAD83 / UTM zone 11N",GEOGCS["NAD83",DATUM["North American Datum 1983",SPHEROID["GRS 1980",6378137,298.257222101004]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",-117],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
NoData: -32768.0

Band descriptions:

  1: constant_B2
  2: constant_B3
  3: constant_B4
  4: constant_B5
  5: constant_B6
  6: constant_B7
  7: constant_B8
  8: constant_B8A
  9: constant_B11
 10: constant_B12
 11: constant_NDVI
 12: constant_NDMI
 13: constant_NDRE
 14: constant_BSI
 15: constant_NBR2
 16: sin1_B2
 17: sin1_B3
 18: sin1_B4
 19: sin1_B5
 20: sin1_B6
 21: sin1_B7
 22: sin1_B8
 23:

In [11]:
with rasterio.open(shard) as src:
    descriptions = list(src.descriptions)

print("\nLast 25 bands:")
for i, desc in list(enumerate(descriptions, start=1))[-25:]:
    print(f"{i:3d}: {desc}")


Last 25 bands:
 67: cos2_B8
 68: cos2_B8A
 69: cos2_B11
 70: cos2_B12
 71: cos2_NDVI
 72: cos2_NDMI
 73: cos2_NDRE
 74: cos2_BSI
 75: cos2_NBR2
 76: nobs
 77: RMSE_B2
 78: RMSE_B3
 79: RMSE_B4
 80: RMSE_B5
 81: RMSE_B6
 82: RMSE_B7
 83: RMSE_B8
 84: RMSE_B8A
 85: RMSE_B11
 86: RMSE_B12
 87: RMSE_NDVI
 88: RMSE_NDMI
 89: RMSE_NDRE
 90: RMSE_BSI
 91: RMSE_NBR2


In [ ]:
# =====================================================================
# QA ALL SOURCE SHARDS AND SAVE AUTHORITATIVE BAND SCHEMAS
# =====================================================================

source_descriptions_by_year = {}
source_lookup_by_year = {}
source_qa_rows = []

required_source_bands = (
    [f"constant_{v}" for v in VARIABLES]
    + [
        f"{trig}{h}_{v}"
        for h in range(1, K + 1)
        for trig in ("sin", "cos")
        for v in VARIABLES
    ]
    + ["nobs"]
    + [f"RMSE_{v}" for v in VARIABLES]
)

assert len(required_source_bands) == EXPECTED_SOURCE_BANDS

for year in YEARS:

    local_shards = local_shards_by_year[year]

    print(f"\nQA {year}")

    qa_rows = []

    for tif in local_shards:

        with rasterio.open(tif) as src:

            qa_rows.append({
                "year": year,
                "filename": tif.name,
                "bands": src.count,
                "crs": str(src.crs),
                "width": src.width,
                "height": src.height,
                "res_x": src.res[0],
                "res_y": src.res[1],
                "nodata": src.nodata,
                "dtype": src.dtypes[0],
                "left": src.bounds.left,
                "bottom": src.bounds.bottom,
                "right": src.bounds.right,
                "top": src.bounds.top,
            })

    qa = pd.DataFrame(qa_rows)

    assert qa["bands"].eq(EXPECTED_SOURCE_BANDS).all(), (
        f"{year}: expected {EXPECTED_SOURCE_BANDS} bands in every shard."
    )

    assert qa["crs"].nunique() == 1, f"{year}: shard CRS mismatch."

    assert qa[["res_x", "res_y"]].drop_duplicates().shape[0] == 1, (
        f"{year}: shard resolution mismatch."
    )

    assert qa["nodata"].eq(NODATA).all(), (
        f"{year}: expected source NoData {NODATA}."
    )

    reference_shard = local_shards[0]

    with rasterio.open(reference_shard) as src:
        source_descriptions = list(src.descriptions)

    if any(d is None for d in source_descriptions):
        raise RuntimeError(f"{year}: reference shard contains unnamed bands.")

    if len(source_descriptions) != EXPECTED_SOURCE_BANDS:
        raise RuntimeError(
            f"{year}: expected {EXPECTED_SOURCE_BANDS} descriptions, "
            f"found {len(source_descriptions)}"
        )

    missing = sorted(
        set(required_source_bands)
        - set(source_descriptions)
    )

    if missing:
        raise RuntimeError(
            f"{year}: missing expected source bands:\n"
            + "\n".join(missing)
        )

    source_descriptions_by_year[year] = source_descriptions

    source_lookup_by_year[year] = {
        name: i
        for i, name in enumerate(source_descriptions, start=1)
    }

    qa.to_csv(
        WORK_ROOT / str(year) / f"{year}_source_shard_QA.csv",
        index=False,
    )

    source_qa_rows.extend(qa_rows)

    print(
        f"{year}: source QA passed — "
        f"{len(source_descriptions)} bands, "
        f"{qa.iloc[0]['crs']}, "
        f"{qa.iloc[0]['res_x']} m"
    )

source_qa = pd.DataFrame(source_qa_rows)

print("\nAll source-shard QA passed.")



QA 2016


RuntimeError: 2016: missing expected source bands:
RMSE_fit_B11
RMSE_fit_B12
RMSE_fit_B2
RMSE_fit_B3
RMSE_fit_B4
RMSE_fit_B5
RMSE_fit_B6
RMSE_fit_B7
RMSE_fit_B8
RMSE_fit_B8A
RMSE_fit_BSI
RMSE_fit_NBR2
RMSE_fit_NDMI
RMSE_fit_NDRE
RMSE_fit_NDVI

In [ ]:
# =====================================================================
# BUILD ALL YEARLY VRTs
# =====================================================================

vrt_by_year = {}

for year in YEARS:

    year_dir = WORK_ROOT / str(year)
    local_shards = local_shards_by_year[year]

    vrt_path = year_dir / f"{year}_K2.vrt"

    print(f"\nBuilding {year}: {vrt_path}")

    cmd = [
        "gdalbuildvrt",
        "-overwrite",
        str(vrt_path),
        *[str(p) for p in local_shards],
    ]

    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
    )

    if result.stdout:
        print(result.stdout)

    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f"{year}: gdalbuildvrt failed.")

    with rasterio.open(vrt_path) as src:

        print("  Bands:", src.count)
        print("  CRS:", src.crs)
        print("  Shape:", src.height, "x", src.width)
        print("  Resolution:", src.res)
        print("  Bounds:", src.bounds)
        print("  Nodata:", src.nodata)

        assert src.count == EXPECTED_SOURCE_BANDS
        assert src.nodata == NODATA

    vrt_by_year[year] = vrt_path

print("\nALL VRTs BUILT SUCCESSFULLY.")


In [ ]:
# =====================================================================
# VRT STATUS TABLE — STOP HERE IF YOU ONLY WANT THE VRT PHASE
# =====================================================================

vrt_status = []

for year in YEARS:

    vrt = WORK_ROOT / str(year) / f"{year}_K2.vrt"

    if not vrt.exists():
        vrt_status.append({
            "year": year,
            "exists": False,
        })
        continue

    with rasterio.open(vrt) as src:
        vrt_status.append({
            "year": year,
            "exists": True,
            "bands": src.count,
            "crs": str(src.crs),
            "width": src.width,
            "height": src.height,
            "res_x": src.res[0],
            "res_y": src.res[1],
            "nodata": src.nodata,
            "path": str(vrt),
        })

vrt_status = pd.DataFrame(vrt_status)

display(vrt_status)

if not vrt_status["exists"].all():
    raise RuntimeError("At least one VRT is missing.")

if not vrt_status["bands"].eq(EXPECTED_SOURCE_BANDS).all():
    raise RuntimeError("At least one VRT has the wrong band count.")

print("\nVRT PHASE COMPLETE.")
print("All 10 yearly VRTs exist and open successfully.")
print("You can stop the notebook here before starting the large TIFF writes.")


# Phase 2 — write all derived K2 monolith TIFFs

This section mirrors the earlier K3 harmonic-sensitivity write.

For K2, the 91 source bands are transformed into 91 output bands:

- 15 intercepts;
- 15 amplitude-1;
- 15 phase-1;
- 15 amplitude-2;
- 15 phase-2;
- 1 `nobs`;
- 15 RMSE bands.

That is:

\[
15 + 30 + 30 + 1 + 15 = 91
\]

The output is written window-by-window so memory use remains bounded.


In [12]:
# =====================================================================
# DEFINE THE DERIVED 91-BAND OUTPUT SCHEMA
# =====================================================================

def build_output_schema():

    schema = []

    # Intercepts
    for variable in VARIABLES:
        schema.append({
            "name": f"constant_{variable}",
            "type": "copy",
            "source": f"constant_{variable}",
        })

    # K2 amplitude + phase
    for h in range(1, K + 1):

        for variable in VARIABLES:
            schema.append({
                "name": f"amp{h}_{variable}",
                "type": "amplitude",
                "sin": f"sin{h}_{variable}",
                "cos": f"cos{h}_{variable}",
            })

        for variable in VARIABLES:
            schema.append({
                "name": f"phase{h}_{variable}",
                "type": "phase",
                "sin": f"sin{h}_{variable}",
                "cos": f"cos{h}_{variable}",
            })

    # QA
    schema.append({
        "name": "nobs",
        "type": "copy",
        "source": "nobs",
    })

    for variable in VARIABLES:
        schema.append({
            "name": f"RMSE_{variable}",
            "type": "copy",
            "source": f"RMSE_{variable}",
        })

    return schema


OUTPUT_SCHEMA = build_output_schema()

print("Output bands:", len(OUTPUT_SCHEMA))

assert len(OUTPUT_SCHEMA) == EXPECTED_OUTPUT_BANDS

display(
    pd.DataFrame(OUTPUT_SCHEMA).head(30)
)


Output bands: 91


,name,type,source,sin,cos
0,constant_B2,copy,constant_B2,NaN,NaN
1,constant_B3,copy,constant_B3,NaN,NaN
2,constant_B4,copy,constant_B4,NaN,NaN
3,constant_B5,copy,constant_B5,NaN,NaN
4,constant_B6,copy,constant_B6,NaN,NaN
5,constant_B7,copy,constant_B7,NaN,NaN
6,constant_B8,copy,constant_B8,NaN,NaN
7,constant_B8A,copy,constant_B8A,NaN,NaN
8,constant_B11,copy,constant_B11,NaN,NaN
9,constant_B12,copy,constant_B12,NaN,NaN


In [13]:
display(pd.DataFrame(OUTPUT_SCHEMA).tail(25))

,name,type,source,sin,cos
66,phase2_B8,phase,NaN,sin2_B8,cos2_B8
67,phase2_B8A,phase,NaN,sin2_B8A,cos2_B8A
68,phase2_B11,phase,NaN,sin2_B11,cos2_B11
69,phase2_B12,phase,NaN,sin2_B12,cos2_B12
70,phase2_NDVI,phase,NaN,sin2_NDVI,cos2_NDVI
71,phase2_NDMI,phase,NaN,sin2_NDMI,cos2_NDMI
72,phase2_NDRE,phase,NaN,sin2_NDRE,cos2_NDRE
73,phase2_BSI,phase,NaN,sin2_BSI,cos2_BSI
74,phase2_NBR2,phase,NaN,sin2_NBR2,cos2_NBR2
75,nobs,copy,nobs,NaN,NaN


In [14]:
# =====================================================================
# HELPERS FOR WINDOWED DERIVED WRITE + ARCHIVE COPY
# =====================================================================

def read_source_band(src, lookup, band_name, window):

    return src.read(
        lookup[band_name],
        window=window,
        masked=True,
    )


def archive_year(year):

    source_dir = WORK_ROOT / str(year)
    archive_dir = ARCHIVE_ROOT / str(year)

    archive_dir.mkdir(parents=True, exist_ok=True)

    print(f"{year}: archiving to {archive_dir}")

    for source in sorted(source_dir.iterdir()):

        if not source.is_file():
            continue

        destination = archive_dir / source.name

        if (
            destination.exists()
            and destination.stat().st_size == source.stat().st_size
        ):
            print("  already archived:", source.name)
            continue

        partial = archive_dir / f"{source.name}.partial"

        if partial.exists():
            partial.unlink()

        shutil.copy2(source, partial)

        if partial.stat().st_size != source.stat().st_size:
            raise RuntimeError(
                f"{year}: archive byte-size mismatch for {source.name}"
            )

        if destination.exists():
            destination.unlink()

        os.replace(partial, destination)

        print("  archived:", source.name)


In [16]:
# =====================================================================
# REBUILD SOURCE BAND LOOKUPS FROM THE AUTHORITATIVE SOURCE SHARDS
# =====================================================================

source_descriptions_by_year = {}
source_lookup_by_year = {}

for year in YEARS:

    # Use the first downloaded source shard as the authoritative band schema.
    shard = local_shards_by_year[year][0]

    with rasterio.open(shard) as src:
        descriptions = list(src.descriptions)

    if len(descriptions) != EXPECTED_SOURCE_BANDS:
        raise RuntimeError(
            f"{year}: expected {EXPECTED_SOURCE_BANDS} source bands, "
            f"found {len(descriptions)}"
        )

    if any(name is None for name in descriptions):
        raise RuntimeError(
            f"{year}: one or more source bands have no description."
        )

    # Make sure names are unique before using them as dictionary keys.
    if len(set(descriptions)) != len(descriptions):
        duplicates = sorted({
            name
            for name in descriptions
            if descriptions.count(name) > 1
        })

        raise RuntimeError(
            f"{year}: duplicate source band names: {duplicates}"
        )

    source_descriptions_by_year[year] = descriptions

    source_lookup_by_year[year] = {
        name: band_number
        for band_number, name in enumerate(
            descriptions,
            start=1,
        )
    }

    print(
        f"{year}: lookup built — "
        f"{len(source_lookup_by_year[year])} bands"
    )

print("\nSource lookups rebuilt for all years.")

2016: lookup built — 91 bands
2017: lookup built — 91 bands
2018: lookup built — 91 bands
2019: lookup built — 91 bands
2020: lookup built — 91 bands
2021: lookup built — 91 bands
2022: lookup built — 91 bands
2023: lookup built — 91 bands
2024: lookup built — 91 bands
2025: lookup built — 91 bands

Source lookups rebuilt for all years.


In [18]:
# =====================================================================
# BUILD / REBUILD ALL MISSING YEARLY VRTs FROM LOCAL SHARDS
# =====================================================================

vrt_by_year = {}

for year in YEARS:

    year_dir = WORK_ROOT / str(year)
    export_prefix = export_prefix_for_year(year)

    # Discover the downloaded GCS shards directly from disk
    local_shards = sorted(
        year_dir.glob(f"{export_prefix}*.tif")
    )

    # Exclude any derived monolith that may already exist
    local_shards = [
        p for p in local_shards
        if "InterceptAmpPhase" not in p.name
        and ".partial." not in p.name
    ]

    print("\n" + "=" * 90)
    print(year)
    print("Local source shards:", len(local_shards))

    if len(local_shards) != 9:
        raise RuntimeError(
            f"{year}: expected 9 source shards, found {len(local_shards)}"
        )

    vrt_path = year_dir / f"{year}_K2.vrt"

    if vrt_path.exists():
        print("VRT already exists:", vrt_path)

    else:
        print("Building:", vrt_path)

        cmd = [
            "gdalbuildvrt",
            "-overwrite",
            str(vrt_path),
            *[str(p) for p in local_shards],
        ]

        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
        )

        if result.stdout:
            print(result.stdout)

        if result.returncode != 0:
            print(result.stderr)
            raise RuntimeError(
                f"{year}: gdalbuildvrt failed"
            )

    # Validate immediately
    with rasterio.open(vrt_path) as src:

        print("Bands:", src.count)
        print("CRS:", src.crs)
        print("Shape:", src.height, "x", src.width)
        print("Resolution:", src.res)
        print("NoData:", src.nodata)

        if src.count != EXPECTED_SOURCE_BANDS:
            raise RuntimeError(
                f"{year}: expected {EXPECTED_SOURCE_BANDS} bands, "
                f"found {src.count}"
            )

        if src.nodata != NODATA:
            raise RuntimeError(
                f"{year}: expected NoData {NODATA}, "
                f"found {src.nodata}"
            )

    vrt_by_year[year] = vrt_path


print("\n" + "#" * 90)
print("ALL YEARLY VRTs COMPLETE")
print("#" * 90)


2016
Local source shards: 9
VRT already exists: C:\NCA_DATA\S2_Harmonics\2016\2016_K2.vrt
Bands: 91
CRS: PROJCS["NAD83 / UTM zone 11N",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",-117],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","26911"]]
Shape: 7857 x 10435
Resolution: (10.0, 10.0)
NoData: -32768.0

2017
Local source shards: 9
Building: C:\NCA_DATA\S2_Harmonics\2017\2017_K2.vrt
0...10...20...30...40...50...60...70...80...90...100 - done.

Bands: 91
CRS: PROJCS["NAD83 / UTM zone 11N",GEOGCS["NAD83",DATUM["North

In [19]:
# =====================================================================
# WRITE ALL FINAL YEARLY K2 MONOLITHS
# =====================================================================

final_products = []

for year in YEARS:

    print("\n" + "#" * 100)
    print(f"WRITING FINAL K2 MONOLITH — {year}")
    print("#" * 100)

    year_dir = WORK_ROOT / str(year)
    vrt_path = year_dir / f"{year}_K2.vrt"

    if not vrt_path.exists():
        raise RuntimeError(f"{year}: VRT missing: {vrt_path}")

    source_lookup = source_lookup_by_year[year]

    final_tif = (
        year_dir
        / f"{year}_K2_InterceptAmpPhase_nobs_RMSE.tif"
    )

    partial_tif = (
        year_dir
        / f"{year}_K2_InterceptAmpPhase_nobs_RMSE.partial.tif"
    )

    band_map_csv = (
        year_dir
        / f"{year}_K2_BandMap.csv"
    )

    success_file = year_dir / "_SUCCESS.txt"

    # Fully complete local product: skip the expensive rewrite.
    if final_tif.exists() and success_file.exists():
        print(f"{year}: final product already complete; skipping write.")

    else:

        if partial_tif.exists():
            partial_tif.unlink()

        with rasterio.open(vrt_path) as src:

            profile = src.profile.copy()

            profile.update(
                driver="GTiff",
                count=len(OUTPUT_SCHEMA),
                dtype="float32",
                nodata=NODATA,
                compress="DEFLATE",
                predictor=3,
                tiled=True,
                blockxsize=512,
                blockysize=512,
                BIGTIFF="YES",
            )

            with rasterio.open(
                partial_tif,
                "w",
                **profile,
            ) as dst:

                for i, band in enumerate(
                    OUTPUT_SCHEMA,
                    start=1,
                ):
                    dst.set_band_description(
                        i,
                        band["name"],
                    )

                dst.update_tags(
                    focal_year=str(year),
                    harmonic_order=str(K),
                    phase_units="radians",
                    phase_definition="atan2(sin_coeff, cos_coeff)",
                    amplitude_definition="sqrt(cos_coeff^2 + sin_coeff^2)",
                    reconstruction=(
                        "constant + sum(amplitude_h * "
                        "cos(h*2*pi*t - phase_h))"
                    ),
                    nobs_definition=(
                        f"valid focal-year {year} observations"
                    ),
                    rmse_definition=(
                        "per-variable RMSE from harmonic regression fit"
                    ),
                )

                windows = list(src.block_windows(1))
                n_windows = len(windows)

                for j, (_, window) in enumerate(
                    windows,
                    start=1,
                ):

                    if (
                        j == 1
                        or j % 50 == 0
                        or j == n_windows
                    ):
                        print(
                            f"{year}: window "
                            f"{j:,} / {n_windows:,}"
                        )

                    out_arrays = []

                    for spec in OUTPUT_SCHEMA:

                        if spec["type"] == "copy":

                            data = read_source_band(
                                src,
                                source_lookup,
                                spec["source"],
                                window,
                            )

                            out = (
                                data
                                .filled(NODATA)
                                .astype(np.float32)
                            )

                        else:

                            sin_arr = read_source_band(
                                src,
                                source_lookup,
                                spec["sin"],
                                window,
                            )

                            cos_arr = read_source_band(
                                src,
                                source_lookup,
                                spec["cos"],
                                window,
                            )

                            combined_mask = (
                                np.ma.getmaskarray(sin_arr)
                                | np.ma.getmaskarray(cos_arr)
                            )

                            sin_data = (
                                sin_arr
                                .filled(np.nan)
                                .astype(np.float32)
                            )

                            cos_data = (
                                cos_arr
                                .filled(np.nan)
                                .astype(np.float32)
                            )

                            if spec["type"] == "amplitude":

                                calc = np.sqrt(
                                    sin_data * sin_data
                                    + cos_data * cos_data
                                )

                            elif spec["type"] == "phase":

                                calc = np.arctan2(
                                    sin_data,
                                    cos_data,
                                )

                            else:
                                raise ValueError(
                                    spec["type"]
                                )

                            calc[combined_mask] = NODATA

                            out = calc.astype(
                                np.float32
                            )

                        out_arrays.append(out)

                    dst.write(
                        np.stack(
                            out_arrays,
                            axis=0,
                        ),
                        window=window,
                    )

        # -------------------------------------------------------------
        # QA PARTIAL PRODUCT BEFORE FINAL RENAME
        # -------------------------------------------------------------

        expected_names = [
            spec["name"]
            for spec in OUTPUT_SCHEMA
        ]

        with rasterio.open(partial_tif) as src:

            assert src.count == EXPECTED_OUTPUT_BANDS
            assert src.nodata == NODATA
            assert list(src.descriptions) == expected_names

            final_crs = str(src.crs)
            final_shape = (src.height, src.width)
            final_res = src.res

        if final_tif.exists():
            final_tif.unlink()

        os.replace(
            partial_tif,
            final_tif,
        )

        # -------------------------------------------------------------
        # BAND MAP
        # -------------------------------------------------------------

        band_map_rows = []

        for i, spec in enumerate(
            OUTPUT_SCHEMA,
            start=1,
        ):

            row = {
                "band": i,
                "band_name": spec["name"],
                "product_type": spec["type"],
                "harmonic_order_model": K,
            }

            if spec["type"] == "copy":
                row["source_band"] = spec["source"]

            else:
                row["source_sin_band"] = spec["sin"]
                row["source_cos_band"] = spec["cos"]

            band_map_rows.append(row)

        band_map = pd.DataFrame(
            band_map_rows
        )

        band_map.to_csv(
            band_map_csv,
            index=False,
        )

        success_file.write_text(
            "\n".join([
                f"year={year}",
                f"final_tif={final_tif}",
                f"bands={EXPECTED_OUTPUT_BANDS}",
                f"nodata={NODATA}",
                f"size_bytes={final_tif.stat().st_size}",
            ]) + "\n",
            encoding="utf-8",
        )

        print(f"{year}: final TIFF QA passed.")

    # -------------------------------------------------------------
    # ARCHIVE COMPLETED YEAR TO A:
    # -------------------------------------------------------------

    archive_year(year)

    archive_final = (
        ARCHIVE_ROOT
        / str(year)
        / final_tif.name
    )

    if (
        not archive_final.exists()
        or archive_final.stat().st_size
        != final_tif.stat().st_size
    ):
        raise RuntimeError(
            f"{year}: archive final-TIFF verification failed."
        )

    final_products.append({
        "year": year,
        "local_tif": str(final_tif),
        "archive_tif": str(archive_final),
        "size_gib": (
            final_tif.stat().st_size
            / 1024**3
        ),
    })

    print(f"{year}: COMPLETE + ARCHIVED")

final_products = pd.DataFrame(final_products)

display(final_products)

print("\nALL FINAL K2 PRODUCTS COMPLETE.")



####################################################################################################
WRITING FINAL K2 MONOLITH — 2016
####################################################################################################
2016: final product already complete; skipping write.
2016: archiving to A:\NCA_DATA\S2_Harmonics\2016
  already archived: 2016_K2.log
  already archived: 2016_K2.vrt
  already archived: 2016_K2_BandMap.csv
  already archived: 2016_K2_InterceptAmpPhase_nobs_RMSE.tif
  already archived: 2016_source_shard_QA.csv
  already archived: _SUCCESS.txt
  already archived: S2SR_Harmonics_2016_K20000000000-0000000000.tif
  already archived: S2SR_Harmonics_2016_K20000000000-0000003584.tif
  already archived: S2SR_Harmonics_2016_K20000000000-0000007168.tif
  already archived: S2SR_Harmonics_2016_K20000003584-0000000000.tif
  already archived: S2SR_Harmonics_2016_K20000003584-0000003584.tif
  already archived: S2SR_Harmonics_2016_K20000003584-0000007168.tif
  already a

,year,local_tif,archive_tif,size_gib
0,2016,C:\NCA_DATA\S2_Harmonics\2016\2016_K2_Intercep...,A:\NCA_DATA\S2_Harmonics\2016\2016_K2_Intercep...,27.306468
1,2017,C:\NCA_DATA\S2_Harmonics\2017\2017_K2_Intercep...,A:\NCA_DATA\S2_Harmonics\2017\2017_K2_Intercep...,27.780384
2,2018,C:\NCA_DATA\S2_Harmonics\2018\2018_K2_Intercep...,A:\NCA_DATA\S2_Harmonics\2018\2018_K2_Intercep...,25.532002
3,2019,C:\NCA_DATA\S2_Harmonics\2019\2019_K2_Intercep...,A:\NCA_DATA\S2_Harmonics\2019\2019_K2_Intercep...,25.636106
4,2020,C:\NCA_DATA\S2_Harmonics\2020\2020_K2_Intercep...,A:\NCA_DATA\S2_Harmonics\2020\2020_K2_Intercep...,25.821216
5,2021,C:\NCA_DATA\S2_Harmonics\2021\2021_K2_Intercep...,A:\NCA_DATA\S2_Harmonics\2021\2021_K2_Intercep...,25.976469
6,2022,C:\NCA_DATA\S2_Harmonics\2022\2022_K2_Intercep...,A:\NCA_DATA\S2_Harmonics\2022\2022_K2_Intercep...,25.984730
7,2023,C:\NCA_DATA\S2_Harmonics\2023\2023_K2_Intercep...,A:\NCA_DATA\S2_Harmonics\2023\2023_K2_Intercep...,25.713758
8,2024,C:\NCA_DATA\S2_Harmonics\2024\2024_K2_Intercep...,A:\NCA_DATA\S2_Harmonics\2024\2024_K2_Intercep...,25.822842
9,2025,C:\NCA_DATA\S2_Harmonics\2025\2025_K2_Intercep...,A:\NCA_DATA\S2_Harmonics\2025\2025_K2_Intercep...,25.423825



ALL FINAL K2 PRODUCTS COMPLETE.


In [20]:
# =====================================================================
# FINAL AUDIT
# =====================================================================

audit_rows = []

for year in YEARS:

    year_dir = WORK_ROOT / str(year)
    archive_dir = ARCHIVE_ROOT / str(year)

    vrt = year_dir / f"{year}_K2.vrt"

    final_tif = (
        year_dir
        / f"{year}_K2_InterceptAmpPhase_nobs_RMSE.tif"
    )

    archive_tif = (
        archive_dir
        / final_tif.name
    )

    row = {
        "year": year,
        "vrt_exists": vrt.exists(),
        "local_tif_exists": final_tif.exists(),
        "archive_tif_exists": archive_tif.exists(),
        "local_success": (year_dir / "_SUCCESS.txt").exists(),
        "archive_success": (archive_dir / "_SUCCESS.txt").exists(),
    }

    if final_tif.exists():
        row["local_size_gib"] = (
            final_tif.stat().st_size
            / 1024**3
        )

    if (
        final_tif.exists()
        and archive_tif.exists()
    ):
        row["archive_size_matches"] = (
            final_tif.stat().st_size
            == archive_tif.stat().st_size
        )

    audit_rows.append(row)

audit = pd.DataFrame(audit_rows)

display(audit)

audit.to_csv(
    WORK_ROOT / "K2_final_audit.csv",
    index=False,
)


,year,vrt_exists,local_tif_exists,archive_tif_exists,local_success,archive_success,local_size_gib,archive_size_matches
0,2016,True,True,True,True,True,27.306468,True
1,2017,True,True,True,True,True,27.780384,True
2,2018,True,True,True,True,True,25.532002,True
3,2019,True,True,True,True,True,25.636106,True
4,2020,True,True,True,True,True,25.821216,True
5,2021,True,True,True,True,True,25.976469,True
6,2022,True,True,True,True,True,25.984730,True
7,2023,True,True,True,True,True,25.713758,True
8,2024,True,True,True,True,True,25.822842,True
9,2025,True,True,True,True,True,25.423825,True


In [21]:
vrt_audit = []

for year in YEARS:

    vrt = WORK_ROOT / str(year) / f"{year}_K2.vrt"

    row = {
        "year": year,
        "exists": vrt.exists(),
        "path": str(vrt),
    }

    if vrt.exists():

        with rasterio.open(vrt) as src:

            row.update({
                "bands": src.count,
                "crs": str(src.crs),
                "width": src.width,
                "height": src.height,
                "res_x": src.res[0],
                "res_y": src.res[1],
                "nodata": src.nodata,
            })

    vrt_audit.append(row)

vrt_audit = pd.DataFrame(vrt_audit)

display(vrt_audit)

,year,exists,path,bands,crs,width,height,res_x,res_y,nodata
0,2016,True,C:\NCA_DATA\S2_Harmonics\2016\2016_K2.vrt,91,"PROJCS[""NAD83 / UTM zone 11N"",GEOGCS[""NAD83"",D...",10435,7857,10.0,10.0,-32768.0
1,2017,True,C:\NCA_DATA\S2_Harmonics\2017\2017_K2.vrt,91,"PROJCS[""NAD83 / UTM zone 11N"",GEOGCS[""NAD83"",D...",10435,7857,10.0,10.0,-32768.0
2,2018,True,C:\NCA_DATA\S2_Harmonics\2018\2018_K2.vrt,91,"PROJCS[""NAD83 / UTM zone 11N"",GEOGCS[""NAD83"",D...",10435,7857,10.0,10.0,-32768.0
3,2019,True,C:\NCA_DATA\S2_Harmonics\2019\2019_K2.vrt,91,"PROJCS[""NAD83 / UTM zone 11N"",GEOGCS[""NAD83"",D...",10435,7857,10.0,10.0,-32768.0
4,2020,True,C:\NCA_DATA\S2_Harmonics\2020\2020_K2.vrt,91,"PROJCS[""NAD83 / UTM zone 11N"",GEOGCS[""NAD83"",D...",10435,7857,10.0,10.0,-32768.0
5,2021,True,C:\NCA_DATA\S2_Harmonics\2021\2021_K2.vrt,91,"PROJCS[""NAD83 / UTM zone 11N"",GEOGCS[""NAD83"",D...",10435,7857,10.0,10.0,-32768.0
6,2022,True,C:\NCA_DATA\S2_Harmonics\2022\2022_K2.vrt,91,"PROJCS[""NAD83 / UTM zone 11N"",GEOGCS[""NAD83"",D...",10435,7857,10.0,10.0,-32768.0
7,2023,True,C:\NCA_DATA\S2_Harmonics\2023\2023_K2.vrt,91,"PROJCS[""NAD83 / UTM zone 11N"",GEOGCS[""NAD83"",D...",10435,7857,10.0,10.0,-32768.0
8,2024,True,C:\NCA_DATA\S2_Harmonics\2024\2024_K2.vrt,91,"PROJCS[""NAD83 / UTM zone 11N"",GEOGCS[""NAD83"",D...",10435,7857,10.0,10.0,-32768.0
9,2025,True,C:\NCA_DATA\S2_Harmonics\2025\2025_K2.vrt,91,"PROJCS[""NAD83 / UTM zone 11N"",GEOGCS[""NAD83"",D...",10435,7857,10.0,10.0,-32768.0


In [22]:
# =====================================================================
# AUDIT RMSE BAND NAMES ACROSS ALL VRTs AND FINAL MONOLITHS
# =====================================================================

rmse_audit_rows = []

for year in YEARS:

    year_dir = WORK_ROOT / str(year)

    vrt_path = year_dir / f"{year}_K2.vrt"

    final_tif = (
        year_dir
        / f"{year}_K2_InterceptAmpPhase_nobs_RMSE.tif"
    )

    # -------------------------------------------------------------
    # VRT
    # -------------------------------------------------------------

    with rasterio.open(vrt_path) as src:

        vrt_descriptions = list(src.descriptions)

        vrt_rmse = [
            name
            for name in vrt_descriptions
            if name is not None
            and str(name).startswith("RMSE")
        ]

    # -------------------------------------------------------------
    # FINAL TIFF
    # -------------------------------------------------------------

    with rasterio.open(final_tif) as src:

        tif_descriptions = list(src.descriptions)

        tif_rmse = [
            name
            for name in tif_descriptions
            if name is not None
            and str(name).startswith("RMSE")
        ]

    rmse_audit_rows.append({
        "year": year,

        "vrt_rmse_count": len(vrt_rmse),
        "vrt_rmse_names": " | ".join(vrt_rmse),

        "tif_rmse_count": len(tif_rmse),
        "tif_rmse_names": " | ".join(tif_rmse),

        "same_rmse_names": vrt_rmse == tif_rmse,
    })


rmse_audit = pd.DataFrame(rmse_audit_rows)

display(rmse_audit)

,year,vrt_rmse_count,vrt_rmse_names,tif_rmse_count,tif_rmse_names,same_rmse_names
0,2016,0,,15,RMSE_B2 | RMSE_B3 | RMSE_B4 | RMSE_B5 | RMSE_B...,False
1,2017,0,,15,RMSE_B2 | RMSE_B3 | RMSE_B4 | RMSE_B5 | RMSE_B...,False
2,2018,0,,15,RMSE_B2 | RMSE_B3 | RMSE_B4 | RMSE_B5 | RMSE_B...,False
3,2019,0,,15,RMSE_B2 | RMSE_B3 | RMSE_B4 | RMSE_B5 | RMSE_B...,False
4,2020,0,,15,RMSE_B2 | RMSE_B3 | RMSE_B4 | RMSE_B5 | RMSE_B...,False
5,2021,0,,15,RMSE_B2 | RMSE_B3 | RMSE_B4 | RMSE_B5 | RMSE_B...,False
6,2022,0,,15,RMSE_B2 | RMSE_B3 | RMSE_B4 | RMSE_B5 | RMSE_B...,False
7,2023,0,,15,RMSE_B2 | RMSE_B3 | RMSE_B4 | RMSE_B5 | RMSE_B...,False
8,2024,0,,15,RMSE_B2 | RMSE_B3 | RMSE_B4 | RMSE_B5 | RMSE_B...,False
9,2025,0,,15,RMSE_B2 | RMSE_B3 | RMSE_B4 | RMSE_B5 | RMSE_B...,False
